In [1]:
import eikon as ek
import pandas as pd
from datetime import datetime, timedelta
import time
ek.set_app_key('7124cc602cee484ab6297f9468d491c9f585afdd')

In [2]:
start_date = (datetime.today() - timedelta(days=1000)).strftime('%Y-%m-%d')
end_date = datetime.today().strftime('%Y-%m-%d')


In [3]:
import pandas as pd
import numpy as np
from ta.trend import MACD, SMAIndicator
from ta.momentum import RSIIndicator
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.volatility import BollingerBands


In [4]:
import eikon as ek
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
from ta.momentum import RSIIndicator
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.volatility import BollingerBands
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# -----------------------------
# Configuration
# -----------------------------
ek.set_app_key("7124cc602cee484ab6297f9468d491c9f585afdd")  # Replace if needed
RIC = 'CLc1'
lookback_minutes = 500
sleep_interval = 60  # seconds

# -----------------------------
# Function: Fetch latest minute data
# -----------------------------
def fetch_data():
    end_time = datetime.utcnow()
    start_time = end_time - timedelta(minutes=lookback_minutes)
    df = ek.get_timeseries(
        RIC,
        interval='minute',
        start_date=start_time,
        end_date=end_time,
        fields=['CLOSE']
    )
    if df is None or df.empty:
        raise ValueError("No data returned.")
    df.rename(columns={'CLOSE': 'Close'}, inplace=True)
    return df.ffill().dropna()

# -----------------------------
# Function: Add Technical Indicators
# -----------------------------
def add_indicators(df):
    df['sma_20'] = SMAIndicator(df['Close'], window=20).sma_indicator()
    df['ema_20'] = EMAIndicator(df['Close'], window=20).ema_indicator()
    df['rsi'] = RSIIndicator(df['Close'], window=14).rsi()
    macd = MACD(df['Close'])
    df['macd'] = macd.macd()
    bb = BollingerBands(df['Close'])
    df['bb_mavg'] = bb.bollinger_mavg()
    df['bb_high'] = bb.bollinger_hband()
    df['bb_low'] = bb.bollinger_lband()
    return df.dropna()

# -----------------------------
# Initial Training (on last 500 minutes)
# -----------------------------
print("Training initial model...")
df = fetch_data()
df = add_indicators(df)
df['future'] = df['Close'].shift(-1)
df['target'] = np.where(
    df['future'].fillna(method='ffill') > df['Close'].fillna(method='ffill'),
    1, 0
)
df.dropna(inplace=True)

# Exclude columns not to be used as features
exclude_cols = ['future', 'target']
feature_cols = [col for col in df.columns if col not in exclude_cols]

X_train = df[feature_cols]
y_train = df['target']
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Model trained.")

C:\Users\ARB\AppData\Local\Temp\ipykernel_8780\2954545294.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = datetime.utcnow()
2025-07-17 17:52:16,825 P[8780] [MainThread 3988] Error code 429 | Client Error: Too many requests, please try again later.
2025-07-17 17:52:16,826 P[8780] [MainThread 3988] HTTP request failed: EikonError-Client Error: Too many requests, please try again later.


Training initial model...


EikonError: Error code 429 | Client Error: Too many requests, please try again later.

In [ ]:
print("Starting real-time prediction...")
plt.ion()  # Turn on interactive mode for live plotting

while True:
    try:
        df_live = fetch_data()
        df_live = add_indicators(df_live)
        latest_row = df_live[feature_cols].iloc[-1:]

        prediction = model.predict(latest_row)[0]
        timestamp = df_live.index[-1]

        print(f"[{timestamp}] Prediction: {'BUY' if prediction == 1 else 'SELL'} | Price: {df_live['Close'].iloc[-1]}")

        # Optional: Plot the latest segment with predictions
        df_live['prediction'] = model.predict(df_live[feature_cols])

        plt.figure(figsize=(14, 6))
        plt.plot(df_live['Close'], label='Close Price', color='black', linewidth=1)
        plt.scatter(df_live[df_live['prediction'] == 1].index,
                    df_live[df_live['prediction'] == 1]['Close'],
                    color='green', label='Buy', marker='^')
        plt.scatter(df_live[df_live['prediction'] == 0].index,
                    df_live[df_live['prediction'] == 0]['Close'],
                    color='red', label='Sell', marker='v')
        plt.title("Live WTI Crude Oil Predictions")
        plt.xlabel("Time")
        plt.ylabel("Price")
        plt.legend()
        plt.tight_layout()
        plt.pause(0.01)
        plt.clf()

        time.sleep(sleep_interval)

    except KeyboardInterrupt:
        print("⛔ Stopped by user.")
        break
    except Exception as e:
        print("⚠️ Error:", e)
        time.sleep(10)